In [ ]:
import json
from pathlib import Path

bm25_path = Path(r"E:\Projects\Numerical Fact Checking\CheckThat 2025\task3\data\1-my dataset\3-top100-BM25.json")
decomp_path = Path(r"E:\Projects\Numerical Fact Checking\CheckThat 2025\task3\data\1-my dataset\3-decomposed-claims.json")
output_path = Path(r"E:\Projects\Numerical Fact Checking\CheckThat 2025\task3\data\1-my dataset\3-concatenated-claims.json")

with bm25_path.open("r", encoding="utf-8") as f:
    bm25_data = json.load(f)

with decomp_path.open("r", encoding="utf-8") as f:
    decomp_data = json.load(f)

decomp_by_claim = {item["original_claim"]: item for item in decomp_data}
decomp_by_id = {item["original_id"]: item for item in decomp_data}

result = []

for item in bm25_data:
    original_id = item["original_id"]
    claim = item["original_claim"]

    decomp_item = decomp_by_claim.get(claim)
    if decomp_item is None:
        decomp_item = decomp_by_id.get(original_id)

    if decomp_item is None:
        raise ValueError(f"No matching decomposed entry found for original_id={original_id}")

    decomposed_questions = decomp_item["decomposed_questions"]
    concatenated_claims = [claim + question for question in decomposed_questions]

    result.append({
        "claim": claim,
        "original_id": original_id,
        "docs": "\n\n".join(item["docs"]),
        "scores": ", ".join(map(str, item["scores"])),
        "concatenated_claims": concatenated_claims,
    })

with output_path.open("w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f"Saved {len(result)} entries to: {output_path}")
print("First concatenated_claims sample:")
for x in result[0]["concatenated_claims"][:5]:
    print(x)
